# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {getattr(metadata, 'identifier', '')}")
print(f"Version: {getattr(metadata, 'version', '')}")
print(f"License: {getattr(metadata, 'license', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

### List all Record Sets
We use the Croissant schema to inspect available Record Sets and their field `@id`s.

In [ ]:
import json

# Show all record sets and fields from the Croissant metadata
def list_record_sets_and_fields(metadata):
    if hasattr(metadata, 'record_sets'):
        for record_set in metadata.record_sets:
            print(f"RecordSet name: {record_set.name}")
            print(f"  @id: {record_set.id}")
            print("  Fields:")
            if hasattr(record_set, 'fields'):
                for field in record_set.fields:
                    field_id = getattr(field, 'id', None)
                    field_name = getattr(field, 'name', None)
                    field_dtype = getattr(field, 'data_type', None)
                    print(f"    - Name: {field_name}")
                    print(f"      @id: {field_id}")
                    print(f"      DataType: {field_dtype}")
            print()
    else:
        print("No record sets found in the metadata.")

list_record_sets_and_fields(metadata)

### Explore Example Records
Below, we fetch and print a handful of the first records from a chosen record set. 

Replace `your_record_set_id` with the desired `@id` from above. All operations are referenced by `@id` as per the Croissant standard.

In [ ]:
# Example: Explore a sample record from a chosen RecordSet
chosen_record_set_id = None

# Iterate and set the first available RecordSet @id
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    chosen_record_set_id = metadata.record_sets[0].id

if chosen_record_set_id is not None:
    print(f"Showing a few records from RecordSet @id = {chosen_record_set_id}\n")
    for i, record in enumerate(dataset.records(record_set=chosen_record_set_id)):
        print(json.dumps(record, indent=2))
        if i >= 2:
            break
else:
    print("No record sets available in this dataset.")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for analysis. All IDs for record sets are dynamically obtained from metadata.

In [ ]:
# Extract data from each record set
dataframes = {}

record_sets_ids = []
if hasattr(metadata, 'record_sets'):
    record_sets_ids = [rs.id for rs in metadata.record_sets]

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from RecordSet @id = {record_set_id}")

if record_sets_ids:
    # Preview fields and data of first record set
    first_rs_id = record_sets_ids[0]
    print(f"\nColumns in RecordSet @id={first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No record sets available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In this section, you will filter, normalize, and group by a field present in the record set. Make sure to reference all entities by their Croissant `@id`.

In [ ]:
# Select a numeric field for analysis
# The field @id, e.g., 'accession_coverage', should come from the record set field overview
import numpy as np

rs_id = None
if record_sets_ids:
    rs_id = record_sets_ids[0]

# Try to autodetect a numeric field
numeric_field_id = None
group_field_id = None
if rs_id is not None and not dataframes[rs_id].empty:
    numeric_candidates = []
    for c in dataframes[rs_id].columns:
        if pd.api.types.is_numeric_dtype(dataframes[rs_id][c]):
            numeric_candidates.append(c)
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0] # Pick first numeric column
        if len(numeric_candidates) > 1:
            group_field_id = numeric_candidates[1] # Possibly group by second numeric
    else:
        # Try to select fields that might contain numbers based on name
        for c in dataframes[rs_id].columns:
            if 'coverage' in c.lower() or 'count' in c.lower() or 'score' in c.lower() or 'mw' in c.lower():
                try:
                    dataframes[rs_id][c] = pd.to_numeric(dataframes[rs_id][c])
                    numeric_field_id = c
                    break
                except Exception:
                    continue
    # Try to use a categorical field for grouping if available
    if group_field_id is None:
        for c in dataframes[rs_id].columns:
            if c != numeric_field_id and dataframes[rs_id][c].dtype == 'object':
                group_field_id = c
                break

if numeric_field_id is not None:
    print(f"Running EDA on RecordSet @id={rs_id} using numeric field '{numeric_field_id}' and group field '{group_field_id}'\n")
    threshold = dataframes[rs_id][numeric_field_id].mean() if dataframes[rs_id][numeric_field_id].dtype != 'O' else None

    if threshold is not None:
        filtered_df = dataframes[rs_id][dataframes[rs_id][numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.4f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id is not None and group_field_id in filtered_df.columns:
            # For numeric group field, bin it into categories
            if pd.api.types.is_numeric_dtype(filtered_df[group_field_id]):
                filtered_df['group_bin'] = pd.qcut(filtered_df[group_field_id], q=3, duplicates='drop')
                group_label = 'group_bin'
            else:
                group_label = group_field_id
            grouped_df = filtered_df.groupby(group_label).mean(numeric_only=True)
            print(f"Grouped data by {group_label}:")
            display(grouped_df.head())
else:
    print("Could not auto-detect a suitable numeric field in the first record set. Please check the columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Visualize the normalized field distribution
if numeric_field_id is not None and 'filtered_df' in locals():
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} (> mean) in RecordSet {rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If there is a grouping variable, visualize the group means
    if group_field_id is not None and group_field_id in filtered_df.columns:
        if pd.api.types.is_numeric_dtype(filtered_df[group_field_id]):
            group_var = 'group_bin'
        else:
            group_var = group_field_id
        group_means = filtered_df.groupby(group_var)[numeric_field_id].mean()
        group_means.plot(kind='bar', figsize=(8,4), title=f"Mean {numeric_field_id} by {group_var}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("No numeric field or filtered DataFrame found for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR^2 dataset containing mass spectrometry analysis of extracellular vesicles isolated from human mast cells using the `mlcroissant` library. By inspecting the Croissant schema, we dynamically listed all record sets and fields with their `@id`, loaded the data into pandas DataFrames, applied basic filtering and normalization on numeric records using field `@id`s, and visualized key data patterns. 

This workflow can be adapted to other Croissant-compliant datasets for reproducible, FAIR-aligned data exploration and further analysis.